# TreeSimplify End-to-End Demo

Full pipeline: beam search + BigInt rational post-processing + multi-pass convergence.

Key features:
- **BigInt rationals**: all expressions parsed with `BigInt(N)//BigInt(D)` to avoid Int64 overflow in polynomial arithmetic
- **Post-processing**: `SymbolicUtils.simplify_fractions` applied to bounded subtrees after beam search
- **Multi-pass convergence**: recursive passes with growing subtree caps (2.5× per pass, up to 3 passes)
- Compares TreeSimplify output vs expected corpus scores

In [ ]:
import Pkg
Pkg.activate(joinpath(pwd(), ".."))
Pkg.instantiate()

In [ ]:
using Symbolics, Latexify, Printf, Random
import TreeSimplify
import TreeSimplify.SymbolicUtils

println("Loaded.")

## Configuration

Default multi-pass pipeline with BigInt rationals.

In [ ]:
config = TreeSimplify.RunConfig()  # uses defaults with BigInt + multi-pass
println("Budget:   max_depth=", config.budget.max_depth,
        " beam_width=", config.budget.beam_width,
        " max_time=", config.budget.max_time_seconds, "s")
println("Post:     max_nodes=", config.post_simplify_max_nodes,
        " timeout=", config.post_simplify_timeout_secs, "s")
println("Passes:   max=", config.simplify_max_passes,
        " growth=", config.simplify_pass_nodes_growth, "×")

## SW Non-RWA Corpus — Full Results

9-section perturbation-theory corpus from superconducting qubit SW interaction.
Comparing TreeSimplify output score vs expected compact-form score.

In [ ]:
function corpus_benchmark()
    sw_path = normpath(joinpath(@__DIR__, "..", "expressions",
        "sw_nonrwa_order4_coeffs.txt"))
    exp_path = normpath(joinpath(@__DIR__, "..", "expressions",
        "extracted_sw_nonrwa_coefficients_output.txt"))
    sw    = TreeSimplify._load_sw_sections(sw_path)
    expec = TreeSimplify._load_expected_sections(exp_path)

    # Run end-to-end validation (input ≡ expected, output ≡ expected)
    e2e = TreeSimplify.run_end_to_end_validation(;
        sw_path = sw_path, expected_path = exp_path, config = config)

    # Collect output scores
    println(rpad("Section", 10), lpad("In", 6), lpad("Out", 6),
            lpad("Exp", 6), lpad("Ratio", 7), rpad("Time", 8),
            "In≡Exp  Out≡Exp")
    println(repeat("-", 70))

    total_ratio = 0.0
    for label in sort!(collect(keys(sw)))
        t = @elapsed result = TreeSimplify.simplify(sw[label]; config = config)
        exp_score = TreeSimplify.expression_score(
            TreeSimplify.expression_term(expec[label]), config.scoring)
        ratio = result.score_after / exp_score
        total_ratio += ratio

        rec = [r for r in e2e.records if r.label == label][1]
        in_eq  = rec.input_equivalent_to_expected  ? "✓" : "✗"
        out_eq = rec.output_equivalent_to_expected ? "✓" : "✗"

        @printf "%-10s %6d %6d %6d %6.2f× %6.1fs  %s     %s\n" (
            label,
            round(Int, result.score_before),
            round(Int, result.score_after),
            round(Int, exp_score),
            ratio, t, in_eq, out_eq)
    end

    avg = total_ratio / 9
    @printf "\nAverage: %.2f× (%.0f%% from target)\n" avg (avg*100-100)
end
corpus_benchmark()

## Hard Test Cases (Regression)

Domain-agnostic algebraic stress tests.

In [ ]:
@variables x y z a b c u v w
cases = [
    ("Cyclic diff-quotient sum",
        ((x^2-y^2)/(x-y)) + ((y^2-z^2)/(y-z)) + ((z^2-x^2)/(z-x))),
    ("Nested rational 3-cycle",
        (1+1/(1+1/(1+x+y))) + (1+1/(1+1/(1+y+z))) + (1+1/(1+1/(1+z+x)))),
    ("High-degree parity cancel",
        (x+y+z)^7 - (x-y-z)^7 + (x+y-z)^7 - (x-y+z)^7),
    ("Squared cyclic diff-quotients",
        ((x^2-y^2)/(x-y))^2 + ((y^2-z^2)/(y-z))^2 + ((z^2-x^2)/(z-x))^2),
    ("CSE-heavy repeated block",
        ((a+b+c)^2 - (a-b-c)^2)*3),
    ("Rewrite-maze neutral elts",
        (((u+0)*1)/1 + ((v+0)*1)/1 + ((w+0)*1)/1) + (u+u) + (v*1) + (w*1)),
    ("Nested cancellation islands",
        (((a+b)/(a+b)) + ((b+c)/(b+c)) + ((c+a)/(c+a)))*1 + 0),
]
println("Loaded ", length(cases), " test cases.")

In [ ]:
for (lbl, expr) in cases
    t = @elapsed r = TreeSimplify.simplify(expr; config = config)
    eq = TreeSimplify.validate_equivalence(expr, r.best_expr, config)
    len_in  = length(TreeSimplify.stable_serialize(expr))
    len_out = length(TreeSimplify.stable_serialize(r.best_expr))

    println(rpad(lbl, 35),
        "  score ", round(Int, r.score_before), " → ", rpad(round(Int, r.score_after), 5),
        "  len ", len_in, " → ", len_out,
        "  ", round(t, digits=2), "s",
        "  eq=", eq.passed,
        "  accepted=", r.accepted)
end

## Determinism Check

Two runs with same config → identical structural hash.

In [ ]:
det_ok = true
for (lbl, expr) in cases
    rA = TreeSimplify.simplify(expr; config = config)
    rB = TreeSimplify.simplify(expr; config = config)
    hA = TreeSimplify.structural_hash(rA.best_expr)
    hB = TreeSimplify.structural_hash(rB.best_expr)
    same = (hA == hB) && (rA.score_after == rB.score_after)
    println(rpad(lbl, 35), " det=", same)
    det_ok &= same
end
println("\nDeterminism passed: ", det_ok)

## Fuzz Sweep

Random algebraic expressions with validation.

In [ ]:
rng = MersenneTwister(0x1234)
vars = [x, y, z]
rand_leaf(rng, vars) = rand(rng) < 0.6 ? vars[rand(rng, 1:length(vars))] : rand(rng, -3:3)

function rand_expr(rng, vars, depth)
    depth <= 0 && return rand_leaf(rng, vars)
    op = rand(rng, [:+, :-, :*, :/])
    a = rand_expr(rng, vars, depth - 1)
    b = rand_expr(rng, vars, depth - 1)
    local b_val = Symbolics.value(b)
    if b_val isa Number && iszero(b_val)
        b = Num(1)
    end
    expr = op === :+ ? (a + b) : op === :- ? (a - b) : op === :* ? (a * b) : (a / b)
    rand(rng) < 0.4 && (expr = (expr + 0) * 1)
    rand(rng) < 0.25 && (expr = expr / 1)
    return expr
end

N = 40
valid = accepted = improved = 0
for i in 1:N
    expr = rand_expr(rng, vars, 3)
    r = TreeSimplify.simplify(expr; config = config)
    rep = TreeSimplify.validate_equivalence(expr, r.best_expr, config)
    valid    += rep.passed ? 1 : 0
    accepted += r.accepted  ? 1 : 0
    improved += (r.score_after < r.score_before) ? 1 : 0
end
println("Runs:       ", N)
println("Validated:  ", valid, "/", N)
println("Accepted:   ", accepted, "/", N)
println("Improved:   ", improved, "/", N)